# Fase A — SFT LoRA del BARC-8B (de-risk de la infra)

Valida que podemos LoRA-fine-tunear el BARC-8B offline en Kaggle GPU con el formato de
training de BARC (chat `messages`, Llama-3.1). Objetivo: la loss baja + el adapter se guarda.
**transformers + peft** (sin unsloth/trl → portable y depurable), imagen ESTANDAR (captura logs),
bf16 en una L4 (24GB), masking manual del prompt (entrena solo el completion del assistant).

In [ ]:
import os, glob, json, time
os.environ['HF_HUB_OFFLINE'] = '1'; os.environ['TRANSFORMERS_OFFLINE'] = '1'
mdir = None
for root, dirs, files in os.walk('/kaggle/input', followlinks=True):
    if 'config.json' in files and any(f.endswith(('.safetensors', '.bin')) for f in files):
        mdir = root; break
assert mdir, 'no se encontro el modelo BARC'
print('modelo en:', mdir)
djson = glob.glob('/kaggle/input/**/induction_heavy_slice.jsonl', recursive=True)
assert djson, 'no se encontro el slice de data'
dpath = djson[0]; print('data en:', dpath)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
MAXLEN = 1536
tok = AutoTokenizer.from_pretrained(mdir, local_files_only=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(mdir, torch_dtype=torch.bfloat16,
                                             device_map='auto', local_files_only=True, attn_implementation='sdpa')
model.config.use_cache = False
lcfg = LoraConfig(r=16, lora_alpha=16, lora_dropout=0.0, bias='none', task_type='CAUSAL_LM',
                  target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
model = get_peft_model(model, lcfg)
model.print_trainable_parameters()
print('chat_template presente:', tok.chat_template is not None)

In [ ]:
from datasets import load_dataset
raw = load_dataset('json', data_files=dpath, split='train')
print('ejemplos crudos:', len(raw))
def tokenize(ex):
    msgs = ex['messages']
    full = tok.apply_chat_template(msgs, tokenize=False)
    prompt = tok.apply_chat_template(msgs[:-1], tokenize=False, add_generation_prompt=True)
    fids = tok(full, truncation=True, max_length=MAXLEN, add_special_tokens=False)['input_ids']
    pids = tok(prompt, truncation=True, max_length=MAXLEN, add_special_tokens=False)['input_ids']
    n_p = min(len(pids), len(fids))
    labels = [-100] * n_p + fids[n_p:]
    return {'input_ids': fids, 'attention_mask': [1] * len(fids), 'labels': labels}
ds = raw.map(tokenize, remove_columns=raw.column_names)
ds = ds.filter(lambda e: len(e['input_ids']) <= MAXLEN and any(l != -100 for l in e['labels']))
print('ejemplos tras tokenizar/filtrar:', len(ds), '| ejemplo len:', len(ds[0]['input_ids']))

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

class DevTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop('labels')
        out = model(**inputs)
        logits = out.logits.float()
        labels = labels.to(logits.device)
        sl = logits[..., :-1, :].contiguous().view(-1, logits.size(-1))
        st = labels[..., 1:].contiguous().view(-1)
        loss = torch.nn.functional.cross_entropy(sl, st, ignore_index=-100)
        return (loss, out) if return_outputs else loss
collator = DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100)
args = TrainingArguments(
    output_dir='/kaggle/working/barc_sft_out', per_device_train_batch_size=1,
    gradient_accumulation_steps=8, warmup_steps=5, max_steps=40, learning_rate=1e-4,
    logging_steps=1, bf16=True, optim='adamw_torch', lr_scheduler_type='cosine',
    seed=42, report_to='none', save_strategy='no')
trainer = DevTrainer(model=model, args=args, train_dataset=ds, data_collator=collator)
t0 = time.time()
stats = trainer.train()
print('SFT OK en', round(time.time()-t0), 's | train_loss:', stats.training_loss)

In [ ]:
model.save_pretrained('/kaggle/working/barc_lora_adapter')
tok.save_pretrained('/kaggle/working/barc_lora_adapter')
losses = [h['loss'] for h in trainer.state.log_history if 'loss' in h]
res = {'n_ejemplos': len(ds), 'max_steps': 40, 'loss_inicial': losses[0] if losses else None,
       'loss_final': losses[-1] if losses else None,
       'bajo': (losses[0] > losses[-1]) if len(losses) > 1 else None}
json.dump(res, open('/kaggle/working/sft_result.json', 'w'), indent=1)
print('SFT DE-RISK:', json.dumps(res))
print('loss:', [round(x, 3) for x in losses])
print('adapter guardado en /kaggle/working/barc_lora_adapter')